# 21 · Full municipal influence refit — Gate 22

Each external municipality is removed once from the validated Bogotá–Region UTAM set. Its incident trips and residents are removed, and the largest strongly connected component, population and sex-specific kernels, supported core-50 targets, and residential distributions are independently reconstructed. Bogotá D.C. remains the fixed comparator.

This is a **point-estimate influence diagnostic**, not a causal intervention or a new bootstrap confidence interval. Gate approval validates reconstruction and completeness, not a desired sign of the result. The unlabelled regional UTAM is retained except in the separate, explicitly labelled mapping diagnostic.


In [1]:
%pip -q install pandas pyarrow numpy scipy matplotlib seaborn geopandas networkx


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, json, os, re, shutil, unicodedata, zipfile

import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path("/content/colombia-stochastic-access")
RAW = ROOT / "data/raw/bogota_em2023"
INTERIM = ROOT / "data/interim/bogota_em2023_gate22"
OUT = ROOT / "outputs/bogota_em2023/gate22"
DRIVE_BASE = Path("/content/drive/MyDrive/colombia-stochastic-access-data/bogota_em2023")
for directory in (RAW, INTERIM, OUT):
    directory.mkdir(parents=True, exist_ok=True)

SOURCE_URL = "https://observatorio.movilidadbogota.gov.co/sites/default/files/2025-09/05_Base%20datos%20procesada%20EODH.zip"
ZONING_URL = "https://observatorio.movilidadbogota.gov.co/sites/default/files/2025-09/03_Zonificacion%20EODH%20%281%29.zip"
PURPOSES = ("employment", "education", "health")
EFFECTS = ("female_minus_male_total", "residential_contribution", "kernel_contribution")
SEX_LEVELS = ("mujer", "hombre")
FLOW_THRESHOLDS = (0.50, 0.80)
TAU = 20.0
BOOTSTRAP_REPLICATES = 500
RANDOM_SEED = 202316
MIN_DESTINATION_SAMPLE = 5
MIN_DESTINATION_ESS = 3.0

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print(f"Entorno fuera de Colab o Drive no montado: {exc}")

G4 = Path(os.environ.get("GATE4_DIR_OVERRIDE", DRIVE_BASE / "results/gate4"))
G6 = Path(os.environ.get("GATE6_DIR_OVERRIDE", DRIVE_BASE / "results/gate6"))
G11 = Path(os.environ.get("GATE11_DIR_OVERRIDE", DRIVE_BASE / "results/gate11"))
G12 = Path(os.environ.get("GATE12_DIR_OVERRIDE", DRIVE_BASE / "results/gate12"))
G13 = Path(os.environ.get("GATE13_DIR_OVERRIDE", DRIVE_BASE / "results/gate13"))
paths = {
    "region_edges": G4 / "transition_edges_utam.parquet",
    "region_targets": G6 / "purpose_target_sets.csv",
    "region_point": G11 / "final_female_male_point_estimates.csv",
    "city_point": G12 / "bogota_dc_internal_female_male_point_estimates.csv",
    "paired_point": G13 / "paired_scope_contrasts_point.csv",
    "gate11": G11 / "gate_11.json",
    "gate12": G12 / "gate_12.json",
    "gate13": G13 / "gate_13.json",
}
missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing validated inputs: " + ", ".join(missing))
for number in (11, 12, 13):
    gate = json.loads(paths[f"gate{number}"].read_text(encoding="utf-8"))
    if not bool(gate.get("gate", {}).get(f"gate_{number}_passed", False)):
        raise RuntimeError(f"Gate {number} failed")
print("Validated dependencies: Gates 11, 12, 13")
print("Output:", OUT)


In [3]:
def valid_zip(path):
    path = Path(path)
    return path.exists() and path.stat().st_size > 0 and zipfile.is_zipfile(path)

def ensure_archive(runtime_name, drive_name, url, label):
    runtime = RAW / runtime_name
    cached = DRIVE_BASE / drive_name
    if valid_zip(runtime):
        return runtime, "runtime_cache"
    if valid_zip(cached):
        shutil.copy2(cached, runtime)
        return runtime, "google_drive_cache"
    from google.colab import files
    print(f"Descargue y seleccione el ZIP de {label}:\n{url}")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Suba exactamente un ZIP")
    name, content = next(iter(uploaded.items()))
    temporary = runtime.with_suffix(".zip.upload")
    temporary.write_bytes(content)
    del content
    if not valid_zip(temporary):
        temporary.unlink(missing_ok=True)
        raise ValueError(f"{name} no es un ZIP válido")
    temporary.replace(runtime)
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    shutil.copy2(runtime, cached)
    return runtime, "manual_upload"

def safe_extract_recursive(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    queue = [(Path(zip_path), destination)]
    visited = set()
    while queue:
        current, target = queue.pop(0)
        signature = (str(current.resolve()), str(target.resolve()))
        if signature in visited:
            continue
        visited.add(signature)
        root = target.resolve()
        with zipfile.ZipFile(current) as zipped:
            for member in zipped.infolist():
                resolved = (target / member.filename).resolve()
                if root != resolved and root not in resolved.parents:
                    raise ValueError(f"Ruta insegura: {member.filename}")
            zipped.extractall(target)
        for nested in target.rglob("*.zip"):
            nested_target = nested.parent / nested.stem
            if (str(nested.resolve()), str(nested_target.resolve())) not in visited:
                queue.append((nested, nested_target))

survey_archive, survey_method = ensure_archive(
    "processed_edoh.zip", "processed_edoh.zip", SOURCE_URL, "microdatos"
)
zoning_archive, zoning_method = ensure_archive(
    "zoning_edoh.zip", "zoning_edoh.zip", ZONING_URL, "zonificación"
)
safe_extract_recursive(survey_archive, INTERIM / "processed_edoh")
safe_extract_recursive(zoning_archive, INTERIM / "zoning_edoh")
print("Microdatos:", survey_method, "| Zonificación:", zoning_method)


Microdatos: runtime_cache | Zonificación: runtime_cache


In [4]:
def normalize(value):
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(char for char in value if not unicodedata.combining(char))
    return re.sub(r"[^a-z0-9]+", "_", value.lower().strip()).strip("_")

def clean_key(series):
    cleaned = series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    return cleaned.mask(cleaned.map(normalize).isin(
        ["", "nan", "none", "na", "no_aplica", "sin_informacion"]
    ))

def canonical_utam(series):
    cleaned = clean_key(series)
    def one(value):
        if pd.isna(value):
            return pd.NA
        text = str(value).strip().upper().replace(" ", "")
        match = re.fullmatch(r"UPR0*(\d+)", text)
        if match:
            return f"UPR{int(match.group(1)):04d}"
        match = re.fullmatch(r"(?:UTAM)?0*(\d+)", text)
        return f"UTAM{int(match.group(1)):03d}" if match else text
    return cleaned.map(one).astype("string")

def parse_number(series):
    raw = series.astype("string").str.strip()
    if float(raw.str.contains(",", regex=False, na=False).mean()) > 0.05:
        raw = raw.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(raw, errors="coerce")

def csv_format(path):
    for encoding in ("utf-8-sig", "latin-1"):
        try:
            with path.open("r", encoding=encoding, errors="strict") as handle:
                sample = handle.read(32768)
            return encoding, csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
        except (UnicodeDecodeError, csv.Error):
            continue
    raise ValueError(f"Formato no detectado: {path}")

def find_module(token, excluded=()):
    candidates = [
        path for path in (INTERIM / "processed_edoh").rglob("*.csv")
        if token in normalize(path.stem)
        and not any(item in normalize(path.stem) for item in excluded)
    ]
    if not candidates:
        raise FileNotFoundError(token)
    return sorted(candidates, key=lambda path: (len(path.parts), len(path.name)))[0]

def read_selected(path, required):
    encoding, delimiter = csv_format(path)
    header = pd.read_csv(path, sep=delimiter, encoding=encoding, nrows=0)
    mapping = {normalize(column): column for column in header.columns}
    absent = sorted(set(required) - set(mapping))
    if absent:
        raise KeyError(f"Faltan columnas {absent}")
    frame = pd.read_csv(
        path, sep=delimiter, encoding=encoding,
        usecols=[mapping[column] for column in required],
        dtype="string", low_memory=False,
    )
    frame.columns = [normalize(column) for column in frame.columns]
    return frame

trips = read_selected(
    find_module("viaje", excluded=("etapa",)),
    ["key_hg", "utam_ori", "utam_des", "fexp_vj", "sexo", "motivo_viaje"],
)
persons = read_selected(
    find_module("persona"),
    ["key_hg", "cod_utam_hg", "fexp_per_5anos", "sexo"],
)
trips["household"] = clean_key(trips["key_hg"])
trips["utam_ori"] = canonical_utam(trips["utam_ori"])
trips["utam_des"] = canonical_utam(trips["utam_des"])
trips["weight"] = parse_number(trips["fexp_vj"])
trips["sex_group"] = trips["sexo"].map(normalize)
trips["purpose_normalized"] = trips["motivo_viaje"].map(normalize)
persons["household"] = clean_key(persons["key_hg"])
persons["residence_utam"] = canonical_utam(persons["cod_utam_hg"])
persons["weight"] = parse_number(persons["fexp_per_5anos"])
persons["sex_group"] = persons["sexo"].map(normalize)
trips = trips[
    trips[["household", "utam_ori", "utam_des"]].notna().all(axis=1)
    & trips["weight"].gt(0)
].reset_index(drop=True)
persons = persons[
    persons[["household", "residence_utam"]].notna().all(axis=1)
    & persons["weight"].gt(0)
].reset_index(drop=True)

PURPOSE_PATTERNS = {
    "employment": r"^a_trabajar$",
    "education": r"estudi|educa|coleg|univers|jardin|escuela",
    "health": r"salud|medic|hospital|clinic|consulta|terapia",
    "job_search": r"buscar_trabajo",
    "mobile_work": r"conduzco_vehiculo_como_forma_de_trabajo",
}
def classify_purpose(value):
    matches = [
        purpose for purpose, pattern in PURPOSE_PATTERNS.items()
        if re.search(pattern, value)
    ]
    return "ambiguous" if len(matches) > 1 else (matches[0] if matches else "other")
trips["purpose_group"] = trips["purpose_normalized"].map(classify_purpose)

spatial_paths = (
    list((INTERIM / "zoning_edoh").rglob("*.shp"))
    + list((INTERIM / "zoning_edoh").rglob("*.geojson"))
    + list((INTERIM / "zoning_edoh").rglob("*.gpkg"))
)
candidates = []
for path in spatial_paths:
    try:
        columns = {normalize(column): column for column in gpd.read_file(path, rows=5).columns}
        if "utam" in columns:
            candidates.append(path)
    except Exception:
        continue
if not candidates:
    raise FileNotFoundError("Capa UTAM")
utam_path = sorted(
    candidates, key=lambda path: ("utam2023" not in normalize(path.stem), len(path.parts))
)[0]
layer = gpd.read_file(utam_path)
column_map = {normalize(column): column for column in layer.columns}
municipality_key = (
    "munnombre" if "munnombre" in column_map
    else next(key for key in column_map if "mun" in key and "nombre" in key)
)
lookup = pd.DataFrame({
    "utam": canonical_utam(layer[column_map["utam"]]),
    "municipality": layer[column_map[municipality_key]].astype("string").str.strip(),
}).dropna().drop_duplicates("utam")
lookup["municipality_normalized"] = lookup["municipality"].map(normalize)
print("Viajes:", len(trips), "| Personas:", len(persons), "| Capa:", utam_path)


Viajes: 97796 | Personas: 64989 | Capa: /content/colombia-stochastic-access/data/interim/bogota_em2023_gate16/zoning_edoh/03_Zonificacion EODH/a. Shapefile UTAM/UTAM2023/UTAM2023.shp


## Municipality support and baseline audits


In [ ]:
edges = pd.read_parquet(paths["region_edges"])
edges["utam_ori"] = canonical_utam(edges["utam_ori"])
edges["utam_des"] = canonical_utam(edges["utam_des"])
region_nodes = sorted(set(edges["utam_ori"]) | set(edges["utam_des"]))
node_set = set(region_nodes)
node_mun = lookup.set_index("utam")["municipality_normalized"].to_dict()
node_label = lookup.set_index("utam")["municipality"].to_dict()
unmapped_nodes = sorted(node_set - set(node_mun))
region_trips = trips[
    trips["utam_ori"].isin(node_set) & trips["utam_des"].isin(node_set)
].copy()
region_persons = persons[persons["residence_utam"].isin(node_set)].copy()
external = sorted({
    node_mun[node] for node in node_set
    if node_mun.get(node) and "bogota" not in node_mun[node]
})
if not external or "facatativa" not in external or "soacha" not in external:
    raise RuntimeError(f"Missing expected external municipalities: {external}")
regional_weights = region_trips["weight"].sum()
residential_weights = region_persons["weight"].sum()
support_rows = []
for municipality in external:
    nodes = {node for node in node_set if node_mun.get(node) == municipality}
    outward = region_trips[region_trips["utam_ori"].isin(nodes)]
    residents = region_persons[region_persons["residence_utam"].isin(nodes)]
    incident = region_trips[
        region_trips["utam_ori"].isin(nodes)
        | region_trips["utam_des"].isin(nodes)
    ]
    row = {
        "municipality": municipality,
        "utam_count": len(nodes),
        "households_outbound": int(outward["household"].nunique()),
        "households_residential": int(residents["household"].nunique()),
        "sampled_outbound_trips": len(outward),
        "weighted_outbound_trips": float(outward["weight"].sum()),
        "weighted_incident_trips": float(incident["weight"].sum()),
        "incident_trip_weight_share": float(incident["weight"].sum() / regional_weights),
        "sampled_residents": len(residents),
        "weighted_residents": float(residents["weight"].sum()),
        "residential_weight_share": float(residents["weight"].sum() / residential_weights),
    }
    for level in ("mujer", "hombre"):
        group = outward[outward["sex_group"].eq(level)]
        weights = group["weight"].to_numpy(dtype=float)
        row[f"{level}_outbound_trip_ess"] = float(
            weights.sum() ** 2 / np.square(weights).sum()
        ) if len(weights) else 0.0
        row[f"{level}_sampled_outbound_trips"] = len(group)
    support_rows.append(row)
municipal_support = pd.DataFrame(support_rows)
municipal_support.to_csv(OUT / "municipality_sample_support.csv", index=False)
display(municipal_support[
    municipal_support["municipality"].isin(["facatativa", "soacha"])
])
print("Regional nodes:", len(region_nodes), "External municipalities:", len(external))
print("Unmapped nodes (kept in every municipal omission):", unmapped_nodes)


## Complete refit after each external municipality is omitted


In [ ]:
def rebuild(removed=None):
    candidates = node_set - {
        node for node in node_set if node_mun.get(node) == removed
    }
    eligible = region_trips[
        region_trips["utam_ori"].isin(candidates)
        & region_trips["utam_des"].isin(candidates)
    ].copy()
    graph = nx.DiGraph()
    grouped = eligible.groupby(["utam_ori", "utam_des"], observed=True)["weight"].sum()
    graph.add_weighted_edges_from(
        (a, b, float(w)) for (a, b), w in grouped.items()
    )
    if graph.number_of_nodes() == 0:
        raise RuntimeError(f"{removed}: empty network")
    component = set(max(nx.strongly_connected_components(graph), key=len))
    nodes = sorted(component)
    index = {node: i for i, node in enumerate(nodes)}
    selected = eligible[
        eligible["utam_ori"].isin(component)
        & eligible["utam_des"].isin(component)
    ].copy()
    residents = region_persons[
        region_persons["residence_utam"].isin(component)
    ].copy()
    n = len(nodes)
    origin = selected["utam_ori"].map(index).to_numpy(dtype=int)
    destination = selected["utam_des"].map(index).to_numpy(dtype=int)
    weight = selected["weight"].to_numpy(dtype=float)
    W0 = np.zeros((n, n), dtype=float)
    np.add.at(W0, (origin, destination), weight)
    totals = W0.sum(axis=1)
    if np.any(totals <= 0):
        raise RuntimeError(f"{removed}: zero row in SCC")
    pool = W0 / totals[:, None]

    purpose_trips = selected[selected["purpose_group"].isin(PURPOSES)].copy()
    purpose_trips["weight_sq"] = purpose_trips["weight"] ** 2
    supported = purpose_trips.groupby(
        ["purpose_group", "utam_des"], observed=True
    ).agg(
        sampled_trips=("weight", "size"),
        weighted_trips=("weight", "sum"),
        sum_weight_sq=("weight_sq", "sum"),
    ).reset_index()
    supported["ess"] = supported["weighted_trips"] ** 2 / supported["sum_weight_sq"]
    supported = supported[
        supported["sampled_trips"].ge(MIN_DESTINATION_SAMPLE)
        & supported["ess"].ge(MIN_DESTINATION_ESS)
    ]
    targets = {}
    for purpose in PURPOSES:
        ranked = supported[supported["purpose_group"].eq(purpose)].sort_values(
            "weighted_trips", ascending=False, kind="stable"
        )
        if ranked.empty:
            raise RuntimeError(f"{removed}/{purpose}: no supported targets")
        cumulative = ranked["weighted_trips"].cumsum() / ranked["weighted_trips"].sum()
        crossing = np.flatnonzero(cumulative.to_numpy(float) >= 0.5)
        count = int(crossing[0] + 1) if len(crossing) else len(ranked)
        targets[purpose] = set(ranked.head(count)["utam_des"])
    kernels = {}
    mus = {}
    for sex in ("mujer", "hombre"):
        selection = selected["sex_group"].eq(sex).fillna(False).to_numpy(bool)
        ww = weight[selection]
        oo, dd = origin[selection], destination[selection]
        W = np.zeros_like(W0)
        np.add.at(W, (oo, dd), ww)
        out = W.sum(axis=1)
        sq = np.bincount(oo, weights=ww ** 2, minlength=n)
        ess = np.divide(out ** 2, sq, out=np.zeros(n), where=sq > 0)
        empirical = np.divide(
            W, out[:, None], out=np.zeros_like(W), where=out[:, None] > 0
        )
        omega = ess / (ess + TAU)
        kernels[sex] = omega[:, None] * empirical + (1 - omega[:, None]) * pool
        group = residents[residents["sex_group"].eq(sex).fillna(False)]
        mu = np.bincount(
            group["residence_utam"].map(index).to_numpy(dtype=int),
            weights=group["weight"].to_numpy(dtype=float),
            minlength=n,
        )
        if mu.sum() <= 0:
            raise RuntimeError(f"{removed}/{sex}: empty residential distribution")
        mus[sex] = mu / mu.sum()

    output = []
    for purpose in PURPOSES:
        target = targets[purpose]
        target_index = np.array(sorted(index[node] for node in target), dtype=int)
        transient = np.array([i for i in range(n) if i not in set(target_index)], dtype=int)
        h = {}
        for sex in ("mujer", "hombre"):
            Q = kernels[sex][np.ix_(transient, transient)]
            hh = np.zeros(n)
            hh[transient] = np.linalg.solve(
                np.eye(len(transient)) - Q, np.ones(len(transient))
            )
            h[sex] = hh
        mu = {}
        for sex in ("mujer", "hombre"):
            distribution = mus[sex].copy()
            distribution[target_index] = 0
            if distribution.sum() <= 0:
                raise RuntimeError(f"{removed}/{purpose}/{sex}: empty outside-target mass")
            mu[sex] = distribution / distribution.sum()
        mm = float(mu["hombre"] @ h["hombre"])
        mw = float(mu["mujer"] @ h["hombre"])
        wm = float(mu["hombre"] @ h["mujer"])
        ww = float(mu["mujer"] @ h["mujer"])
        residential = (mw - mm + ww - wm) / 2
        kernel = (wm - mm + ww - mw) / 2
        gap = ww - mm
        output.append({
            "excluded_municipality": removed or "<none>",
            "purpose_group": purpose,
            "nodes_in_scc": n,
            "trip_rows": len(selected),
            "residential_rows": len(residents),
            "trip_weight_scc_coverage": float(
                selected["weight"].sum() / eligible["weight"].sum()
            ),
            "target_nodes": len(target),
            "target_set": ";".join(sorted(target)),
            "male_total_fpt": mm,
            "female_total_fpt": ww,
            "female_minus_male_total": gap,
            "residential_contribution": residential,
            "kernel_contribution": kernel,
            "relative_gap": gap / ((ww + mm) / 2),
            "closure_error": gap - residential - kernel,
        })
    return pd.DataFrame(output), nodes, pool, targets


In [ ]:
baseline, baseline_nodes, baseline_pool, baseline_targets = rebuild()
regional_verified = pd.read_csv(paths["region_point"])
city_verified = pd.read_csv(paths["city_point"])
paired_verified = pd.read_csv(paths["paired_point"])
check = baseline.merge(
    regional_verified, on="purpose_group", suffixes=("_new", "_gate11"),
    validate="one_to_one",
)
effects = ("female_minus_male_total", "residential_contribution", "kernel_contribution")
baseline_error = max(
    float((check[f"{key}_new"] - check[f"{key}_gate11"]).abs().max())
    for key in effects
)
reference_index = {node: i for i, node in enumerate(region_nodes)}
reference_pool = np.zeros((len(region_nodes), len(region_nodes)))
for edge in edges.itertuples(index=False):
    reference_pool[
        reference_index[edge.utam_ori], reference_index[edge.utam_des]
    ] = float(edge.transition_probability)
pool_error = (
    float(np.max(np.abs(baseline_pool - reference_pool)))
    if baseline_nodes == region_nodes else float("inf")
)
saved_targets = pd.read_csv(paths["region_targets"], dtype={"utam": "string"})
saved_targets["utam"] = canonical_utam(saved_targets["utam"])
target_match = {
    purpose: baseline_targets[purpose] == set(
        saved_targets.loc[
            saved_targets["purpose_group"].eq(purpose)
            & saved_targets["target_definition"].eq("core_50"), "utam"
        ]
    )
    for purpose in PURPOSES
}
if baseline_error > 1e-8 or pool_error > 1e-10 or not all(target_match.values()):
    raise RuntimeError(
        f"Baseline mismatch: gap={baseline_error}, pool={pool_error}, targets={target_match}"
    )
print("Validated baseline errors:", baseline_error, pool_error)
rows = [baseline]
failures = []
for municipality in external:
    try:
        estimate, _, _, _ = rebuild(municipality)
        rows.append(estimate)
        print("Refitted:", municipality)
    except Exception as exc:
        failures.append({"municipality": municipality, "error": repr(exc)})
        print("FAILED:", municipality, repr(exc))
refits = pd.concat(rows, ignore_index=True)
refits.to_csv(OUT / "municipality_full_refit_point_estimates.csv", index=False)
pd.DataFrame(failures, columns=["municipality", "error"]).to_csv(
    OUT / "municipality_full_refit_failures.csv", index=False
)
if failures:
    raise RuntimeError("Some municipal refits failed; inspect municipality_full_refit_failures.csv")


## Paired influence, sample diagnostics and independent gate


In [ ]:
city = city_verified.set_index("purpose_group")
reference = baseline.set_index("purpose_group")
observed_paired = paired_verified.set_index("purpose_group")
influence = refits[refits["excluded_municipality"].ne("<none>")].copy()
influence["district_gap"] = influence["purpose_group"].map(
    city["female_minus_male_total"]
)
influence["baseline_regional_gap"] = influence["purpose_group"].map(
    reference["female_minus_male_total"]
)
influence["baseline_paired_gap"] = (
    influence["baseline_regional_gap"] - influence["district_gap"]
)
influence["refit_paired_gap"] = (
    influence["female_minus_male_total"] - influence["district_gap"]
)
influence["change_in_paired_gap"] = (
    influence["refit_paired_gap"] - influence["baseline_paired_gap"]
)
influence["baseline_relative_gap"] = influence["purpose_group"].map(
    reference["relative_gap"]
)
influence["district_relative_gap"] = (
    city["female_minus_male_total"]
    / ((city["female_total_fpt"] + city["male_total_fpt"]) / 2)
).reindex(influence["purpose_group"]).to_numpy()
influence["refit_relative_scope_contrast"] = (
    influence["relative_gap"] - influence["district_relative_gap"]
)
influence = influence.merge(municipal_support, left_on="excluded_municipality",
                            right_on="municipality", validate="many_to_one")
influence.to_csv(OUT / "municipality_full_refit_influence.csv", index=False)
paired_error = max(
    abs(
        float(reference.loc[p, "female_minus_male_total"])
        - float(city.loc[p, "female_minus_male_total"])
        - float(observed_paired.loc[p, "region_minus_city__female_minus_male_total"])
    )
    for p in PURPOSES
)
summary = influence.groupby("purpose_group", observed=True).agg(
    min_refit_paired_gap=("refit_paired_gap", "min"),
    max_refit_paired_gap=("refit_paired_gap", "max"),
    min_refit_relative_contrast=("refit_relative_scope_contrast", "min"),
    max_abs_change=("change_in_paired_gap", lambda x: x.abs().max()),
).reset_index()
summary.to_csv(OUT / "municipality_full_refit_summary.csv", index=False)
display(summary.round(4))
display(influence[influence["excluded_municipality"].isin(["facatativa", "soacha"])][[
    "excluded_municipality", "purpose_group", "households_outbound",
    "mujer_outbound_trip_ess", "hombre_outbound_trip_ess",
    "nodes_in_scc", "target_nodes", "refit_paired_gap",
    "change_in_paired_gap", "refit_relative_scope_contrast",
]].round(4))
audit = {
    "baseline_region_points_reproduced": bool(baseline_error <= 1e-8),
    "baseline_population_kernel_reproduced": bool(pool_error <= 1e-10),
    "baseline_targets_reproduced": bool(all(target_match.values())),
    "paired_gate13_reproduced": bool(paired_error <= 1e-8),
    "all_external_municipalities_refitted": bool(
        set(influence["excluded_municipality"]) == set(external)
        and len(influence) == len(external) * len(PURPOSES)
    ),
    "complete_finite_output": bool(np.isfinite(
        influence.select_dtypes(include="number").to_numpy()
    ).all()),
    "decomposition_closed": bool(refits["closure_error"].abs().max() <= 1e-10),
    "facatativa_and_soacha_present": bool(
        {"facatativa", "soacha"} <= set(influence["excluded_municipality"])
    ),
}
diagnostics = {
    "all_refit_paired_gaps_positive": bool(influence["refit_paired_gap"].gt(0).all()),
    "all_refit_relative_contrasts_positive": bool(
        influence["refit_relative_scope_contrast"].gt(0).all()
    ),
}
audit["gate_22_passed"] = bool(all(audit.values()))
report = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "municipalities_tested": len(external),
    "unmapped_nodes_retained": unmapped_nodes,
    "baseline_replication_error": baseline_error,
    "baseline_kernel_error": pool_error,
    "paired_replication_error": paired_error,
    "gate": audit,
    "substantive_diagnostics": diagnostics,
    "summary": json.loads(summary.to_json(orient="records")),
}
(OUT / "gate_22.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False),
    encoding="utf-8"
)
display(pd.Series({**audit, **diagnostics}, name="result"))
if not audit["gate_22_passed"]:
    raise RuntimeError("Gate 22 failed; inspect outputs")
persistent = DRIVE_BASE / "results/gate22"
persistent.mkdir(parents=True, exist_ok=True)
for artifact in OUT.iterdir():
    if artifact.is_file():
        shutil.copy2(artifact, persistent / artifact.name)
print("Gate 22: APPROVED")
print("Persistent copy:", persistent)
